# Seção 6 — Conclusões e Recomendações Práticas

| | |
|---|---|
| **Projeto** | Análise Estatística — Spaceship Titanic (Pós-Graduação em Deep Learning — CIn/UFPE) |
| **Seção T3** | 6. Conclusões e Recomendações Práticas |
| **Ticket** | [KAN-44](https://megamatic.atlassian.net/browse/KAN-44) — fechamento das pendências de conteúdo e consolidação |
| **Responsável** | Gustavo Menezes Gonçalves |
| **Revisora** | Daniela Amaro |
| **Base de dados** | `data/processed/spaceship_titanic_clean.csv` — 8.693 passageiros × 22 colunas, 0 valores ausentes |
| **Insumos** | Seções 1 a 5 do notebook consolidado e os documentos `docs/conclusoes/01` a `05` |

> **Papel deste capítulo.** Ele **não produz análise nova**. Reúne os números já estabelecidos nas Seções 1 a 5, confere uma parte deles contra a base limpa e os traduz em recomendações compreensíveis por quem não tem formação técnica. Toda afirmação aponta a seção de origem, para que o relatório (KAN-39) possa rastreá-la.

## 1. Como ler este capítulo

O projeto investigou **o que distingue os passageiros transportados dos não transportados** no incidente da *Spaceship Titanic*. A resposta tem três camadas, e a ordem importa:

1. **Quem era o passageiro** — origem, cabine, idade, estado de criossono;
2. **Como se comportou a bordo** — consumo nas cinco rubricas de gasto;
3. **O que o conjunto das variáveis consegue prever** — modelo logístico e sua avaliação fora da amostra.

A primeira célula recomputa os indicadores de manchete a partir da base limpa. Os valores servem de conferência: se a base mudar, os números abaixo deixam de bater e a célula final falha, em vez de produzir um resumo desatualizado.

In [1]:
# =============================================================================
# Painel de evidências da Seção 6
#
# Recomputa apenas indicadores de manchete já analisados nas Seções 1 a 5.
# Nenhuma análise nova: o objetivo é conferir os números citados no texto.
# =============================================================================
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(42)  # semente fixa do projeto, declarada literalmente

CANDIDATOS = [
    Path("../data/processed/spaceship_titanic_clean.csv"),
    Path("data/processed/spaceship_titanic_clean.csv"),
]
DATA_PATH = next((caminho for caminho in CANDIDATOS if caminho.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Base tratada não encontrada a partir de notebooks/ ou da raiz.")

df = pd.read_csv(DATA_PATH)


def pct(valor: float) -> str:
    """Formata proporção (0-1) como percentual com duas casas e vírgula decimal."""
    return f"{100 * valor:.2f}%".replace(".", ",")


def brl(valor: float) -> str:
    """Formata valor monetário no padrão brasileiro."""
    return f"US$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def milhar(valor: int) -> str:
    """Formata inteiro com separador de milhar brasileiro."""
    return f"{valor:,}".replace(",", ".")


# A base usa os nomes em inglês; o texto do capítulo usa os nomes em português.
ROTULO_PLANETA = {"Earth": "Terra", "Mars": "Marte", "Europa": "Europa"}


taxa_global = df["Transported"].mean()
por_planeta = df.groupby("HomePlanet")["Transported"].mean()
por_criossono = df.groupby("CryoSleep")["Transported"].mean()
zero_gasto = (df["TotalSpend"] == 0).mean()
mediana_por_transporte = df.groupby("Transported")["TotalSpend"].median()
media_por_planeta = df.groupby("HomePlanet")["TotalSpend"].mean()
mediana_por_planeta = df.groupby("HomePlanet")["TotalSpend"].median()

print(f"Taxa global de transporte: {pct(taxa_global)} ({milhar(int(df['Transported'].sum()))} de {milhar(len(df))})")
print()
print("Taxa de transporte por planeta de origem:")
for planeta, valor in por_planeta.sort_values(ascending=False).items():
    print(f"  {ROTULO_PLANETA.get(planeta, planeta):<7} {pct(valor)}")
print()
print("Taxa de transporte por estado de criossono:")
for estado, valor in por_criossono.items():
    rotulo = "Em criossono" if estado else "Acordado"
    print(f"  {rotulo:<14} {pct(valor)}")
print()
print(f"Passageiros sem nenhum gasto: {pct(zero_gasto)} ({milhar(int((df['TotalSpend'] == 0).sum()))})")
print()
print("Gasto total por desfecho:")
for transportado, valor in mediana_por_transporte.items():
    rotulo = "Transportado" if transportado else "Não transportado"
    print(f"  {rotulo:<17} mediana {brl(valor)}")
print()
print("Gasto total por planeta (média e mediana):")
for planeta in media_por_planeta.sort_values(ascending=False).index:
    print(f"  {ROTULO_PLANETA.get(planeta, planeta):<7} média {brl(media_por_planeta[planeta])} | mediana {brl(mediana_por_planeta[planeta])}")

Taxa global de transporte: 50,36% (4.378 de 8.693)

Taxa de transporte por planeta de origem:
  Europa  66,05%
  Marte   52,44%
  Terra   42,33%

Taxa de transporte por estado de criossono:
  Acordado       32,82%
  Em criossono   81,47%

Passageiros sem nenhum gasto: 41,92% (3.644)

Gasto total por desfecho:
  Não transportado  mediana US$ 907,00
  Transportado      mediana US$ 0,00

Gasto total por planeta (média e mediana):
  Europa  média US$ 3.444,00 | mediana US$ 1.914,00
  Marte   média US$ 1.059,73 | mediana US$ 950,00
  Terra   média US$ 675,24 | mediana US$ 705,00


## 2. Painel de evidências

| Indicador | Valor | Origem |
| :--- | ---: | :--- |
| Taxa global de transporte | **50,36%** (4.378 de 8.693) | [§2], `docs/conclusoes/02` |
| Transporte por origem — Europa | **66,05%** | [§2], `fig_01` |
| Transporte por origem — Marte | **52,44%** | [§2], `fig_01` |
| Transporte por origem — Terra | **42,33%** | [§2], `fig_01` |
| Transporte em criossono × acordado | **81,47% × 32,82%** | [§2], `fig_02` |
| Associação `HomePlanet` × `Transported` | $\chi^2 = 338{,}82$; $V = 0{,}197$ | Seção 4, `04` |
| Associação `CryoSleep` × `Transported` | $\chi^2 = 1.897{,}71$; $V = 0{,}467$ | Seção 4, `04` |
| Passageiros sem nenhum gasto | **41,92%** (3.644) | [§3], `fig_08` |
| Gasto mediano por desfecho | **US$ 0,00** (transportado) × **US$ 907,00** (não) | Seção 4, `04` |
| Gasto médio por origem — Europa | **US$ 3.444,00** | Seção 4, `04` |
| Idade — aderência à Normal | W = 0,980; D = 0,076 | Seção 3, `03` |
| Gastos positivos — aderência à Log-Normal | W = 0,891; D = 0,126 | Seção 3, `03` |
| Modelo logístico — acurácia no teste | **77,9%** | Seção 5, `05` |
| Modelo logístico — ROC-AUC | **0,862** | Seção 5, `05` |

**Leitura em uma frase.** O estado de criossono é a variável isolada mais associada ao transporte; a origem planetária e o consumo vêm em seguida; e o conjunto das variáveis permite prever o desfecho com 77,9% de acurácia, bem acima dos 50,4% de uma regra ingênua.

## 3. Recomendações práticas

As recomendações abaixo são **de triagem e de investigação**, não de causalidade. Nenhuma delas afirma que uma variável *causou* o transporte.

1. **Priorizar a inspeção das cápsulas de criossono.** Passageiros em criossono foram transportados em 81,47% dos casos, contra 32,82% entre os acordados. É a associação mais forte do estudo (V de Cramér = 0,467, muito forte). Ação: tratar as cápsulas como área crítica na apuração e na comunicação às famílias.
2. **Investigar a diferença por origem planetária.** Europa (66,05%) e Marte (52,44%) têm taxas bem acima da Terra (42,33%), com associação estatisticamente significativa ($\chi^2 = 338{,}82$; $p < 10^{-73}$). Ação: cruzar origem com deck e classe da cabine para entender se o efeito é de localização física a bordo. **Não** usar origem como critério isolado de priorização.
3. **Não tratar gasto como proxy de importância ou de presença.** 41,92% dos passageiros não gastaram nada, e a mediana dos transportados é zero. Ação: usar cabine, grupo de viagem e estado de criossono para reconstruir a alocação a bordo; gasto serve para caracterizar perfil de consumo, não para inferir risco.
4. **Usar o modelo como apoio à triagem, com revisão humana.** O modelo logístico acerta 77,9% no teste, com ROC-AUC de 0,862 e ganho de 25,5 p.p. sobre a regra ingênua. Mas o limiar de 0,5 foi fixado *a priori* e o recall é de 0,796. Ação: recalibrar o limiar conforme o custo relativo de falso positivo e falso negativo antes de qualquer uso operacional.
5. **Comunicar com medianas e proporções, não com médias.** A média de gasto é inflada pela cauda longa (assimetria 4,40). Ação: nos relatórios ao cliente, usar mediana, proporção de gasto zero e taxa de transporte por grupo.

## 4. Limitações

- **Associação não é causa.** Todas as diferenças são observacionais; o desenho não permite isolar mecanismos.
- **Divisão única de treino e teste**, sem intervalo de incerteza para as métricas.
- **Divisão por passageiro, não por grupo de viagem.** Famílias podem aparecer nos dois lados, o que tende a deixar a avaliação otimista em magnitude desconhecida.
- **Imputação pela moda** em 2,1% a 2,5% das categóricas; a sensibilidade já documentada chega a 0,64 p.p. e não muda as conclusões.
- **Limiar 0,5 não otimizado.** Outros limiares trocam recall por especificidade.
- **`HomePlanet_Europa` com VIF de 7,98**, pendência registrada no KAN-31; os coeficientes dessa variável devem ser lidos com cautela.

## 5. Próximos passos

| Etapa | Ticket | O que consome deste capítulo |
| :--- | :--- | :--- |
| Relatório técnico de 10 a 15 páginas | KAN-39 | Recomendações 1 a 5 e o painel de evidências |
| Revisão cruzada | KAN-40 | Conferência dos números contra as seções de origem |
| Validação numérica | KAN-41 | Reprodução do painel pela célula de código |
| PDF e pacote final | KAN-42 / KAN-43 | Texto final e figuras já catalogadas |

**O que não fazer:** usar este capítulo como fonte primária de números. Ele é uma síntese; a fonte é a seção indicada em cada linha do painel.

In [2]:
# =============================================================================
# Guarda-corpo do capítulo
#
# Se a base mudar, estes números deixam de bater e a célula falha: o resumo
# executivo não pode ficar desatualizado em silêncio.
# =============================================================================
assert len(df) == 8693, "A base tratada deve ter 8.693 passageiros."
assert int(df["Transported"].sum()) == 4378, "Esperados 4.378 transportados."
assert int((df["TotalSpend"] == 0).sum()) == 3644, "Esperados 3.644 passageiros sem gasto."
assert abs(taxa_global - 0.5036) < 0.0001, "Taxa global fora do intervalo esperado."
print("Checagens do capítulo: OK")

Checagens do capítulo: OK
